In [1]:
import pandas as pd

GLOBI_PATH = "/scratch/ariana.l/CfE2026CVforEcology/rawpollinatordata/interactions.csv.gz"

# Read just the first 2 rows to inspect columns
sample = pd.read_csv(GLOBI_PATH, nrows=2, low_memory=False)
print(f"Total columns: {len(sample.columns)}")
print("\nAll column names:")
for col in sample.columns:
    print(f"  {col}")

Total columns: 92

All column names:
  sourceTaxonId
  sourceTaxonIds
  sourceTaxonName
  sourceTaxonRank
  sourceTaxonPathNames
  sourceTaxonPathIds
  sourceTaxonPathRankNames
  sourceTaxonSpeciesName
  sourceTaxonSpeciesId
  sourceTaxonSubgenusName
  sourceTaxonSubgenusId
  sourceTaxonGenusName
  sourceTaxonGenusId
  sourceTaxonFamilyName
  sourceTaxonFamilyId
  sourceTaxonOrderName
  sourceTaxonOrderId
  sourceTaxonClassName
  sourceTaxonClassId
  sourceTaxonPhylumName
  sourceTaxonPhylumId
  sourceTaxonKingdomName
  sourceTaxonKingdomId
  sourceId
  sourceOccurrenceId
  sourceInstitutionCode
  sourceCollectionCode
  sourceCatalogNumber
  sourceBasisOfRecordId
  sourceBasisOfRecordName
  sourceLifeStageId
  sourceLifeStageName
  sourceBodyPartId
  sourceBodyPartName
  sourcePhysiologicalStateId
  sourcePhysiologicalStateName
  sourceSexId
  sourceSexName
  interactionTypeName
  interactionTypeId
  targetTaxonId
  targetTaxonIds
  targetTaxonName
  targetTaxonRank
  targetTaxonPathNa

In [ ]:
#Extracting the interactions from CONUS

import pandas as pd

GLOBI_PATH = "/scratch/ariana.l/CfE2026CVforEcology/rawpollinatordata/interactions.csv.gz"
OUT_PATH = "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_globi_conus_broad.csv"

BROAD_INTERACTION_TYPES = {
    'pollinates', 'pollinatedBy',
    'visitsFlowersOf', 'flowersVisitedBy',
    'visits', 'visitedBy',
    'hasFlowerVisitor'
}

CONUS_LON = (-125, -66)
CONUS_LAT  = (24, 50)

KEEP_COLS = [
    'sourceTaxonName', 'sourceTaxonOrderName', 'sourceTaxonFamilyName',
    'targetTaxonName', 'targetTaxonOrderName', 'targetTaxonFamilyName',
    'interactionTypeName',
    'decimalLatitude', 'decimalLongitude'
]

chunks = []
for chunk in pd.read_csv(GLOBI_PATH, usecols=KEEP_COLS, chunksize=100_000, low_memory=False):
    # interaction type filter
    chunk = chunk[chunk['interactionTypeName'].isin(BROAD_INTERACTION_TYPES)]
    # CONUS filter (drop rows with missing coords first)
    chunk = chunk.dropna(subset=['decimalLatitude', 'decimalLongitude'])
    chunk = chunk[
        chunk['decimalLongitude'].between(*CONUS_LON) &
        chunk['decimalLatitude'].between(*CONUS_LAT)
    ]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
df.to_csv(OUT_PATH, index=False)

print(f"Saved {len(df):,} rows to {OUT_PATH}")
print(f"\nInteraction type breakdown:")
print(df['interactionTypeName'].value_counts())

In [ ]:
import subprocess
result = subprocess.run(['find', '/scratch/ariana.l', '-name', 'pollinator_observations*.csv'], 
                       capture_output=True, text=True)
print(result.stdout)

In [ ]:
#check GloBI pollinator species coverage against GBIF
import pandas as pd

GBIF_PATH = "/scratch/ariana.l/Plant Pollinator Initial Analysis/pollinator_observations_v2.csv"
GLOBI_PATH = "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_globi_conus_broad.csv"

globi = pd.read_csv(GLOBI_PATH)

# Get unique species from both sides of GloBI interactions
globi_species = set(globi['sourceTaxonName'].dropna()) | set(globi['targetTaxonName'].dropna())
print(f"Unique taxa in GloBI CONUS broad: {len(globi_species):,}")

# Get unique species in GBIF
gbif_species = set(pd.read_csv(GBIF_PATH, usecols=['pollinator_species'])['pollinator_species'].dropna())
print(f"Unique pollinator species in GBIF: {len(gbif_species):,}")

# Overlap
overlap = globi_species & gbif_species
print(f"Species in both: {len(overlap):,}")
print(f"GloBI taxa missing from GBIF: {len(globi_species - gbif_species):,}")

In [5]:
#Check what are the taxonomic order group that are included in the GLOBI interactions that are not inluded in the GBIF initial download
source_is_pollinator = {'visitsFlowersOf', 'pollinates', 'visits', 'hasFlowerVisitor'}
target_is_pollinator = {'flowersVisitedBy', 'pollinatedBy', 'visitedBy'}

pol_source = globi.loc[globi['interactionTypeName'].isin(source_is_pollinator), 
                        ['sourceTaxonName', 'sourceTaxonOrderName']].rename(
                        columns={'sourceTaxonName': 'taxon', 'sourceTaxonOrderName': 'order'})

pol_target = globi.loc[globi['interactionTypeName'].isin(target_is_pollinator), 
                        ['targetTaxonName', 'targetTaxonOrderName']].rename(
                        columns={'targetTaxonName': 'taxon', 'targetTaxonOrderName': 'order'})

pol_all = pd.concat([pol_source, pol_target]).drop_duplicates(subset='taxon')

print("Pollinator order breakdown in GloBI CONUS:")
print(pol_all['order'].value_counts(dropna=False).head(20))

Pollinator order breakdown in GloBI CONUS:
order
Hymenoptera       2821
Lepidoptera       2079
Diptera           1333
Coleoptera        1267
Hemiptera          813
NaN                753
Araneae            189
Passeriformes      173
Asterales          145
Orthoptera         123
Odonata             85
Lamiales            69
Rosales             36
Caryophyllales      34
Gentianales         30
Fabales             29
Trombidiformes      23
Ericales            23
Rodentia            23
Apodiformes         20
Name: count, dtype: int64


In [6]:
# Orders already in our GBIF download
GBIF_COVERED = {'Hymenoptera', 'Lepidoptera', 'Diptera', 'Coleoptera', 'Apodiformes'}

# Plant orders (GloBI data quality errors, not missing pollinators)
PLANT_ORDERS = {'Asterales', 'Lamiales', 'Rosales', 'Caryophyllales', 
                'Gentianales', 'Fabales', 'Ericales', 'Proteales', 
                'Solanales', 'Malpighiales'}

all_orders = pol_all['order'].value_counts(dropna=False)

missing = all_orders[
    ~all_orders.index.isin(GBIF_COVERED) & 
    ~all_orders.index.isin(PLANT_ORDERS) &
    all_orders.index.notna()
]

print("Orders to download from GBIF:")
print(missing.to_string())
print(f"\nTotal missing orders: {len(missing)}")

Orders to download from GBIF:
order
Hemiptera                    813
Araneae                      189
Passeriformes                173
Orthoptera                   123
Odonata                       85
Trombidiformes                23
Rodentia                      23
Squamata                      20
Asparagales                   19
Neuroptera                    18
Piciformes                    17
Stylommatophora               17
Psocodea                      16
Apiales                       16
Agaricales                    16
Anura                         15
Liliales                      15
Ranunculales                  15
Boraginales                   14
Dipsacales                    11
Polypodiales                  11
Brassicales                   11
Blattodea                     11
Mantodea                      10
Thysanoptera                   9
Helotiales                     9
Chiroptera                     9
Sapindales                     9
Pucciniales                    8
Collemb

In [7]:
import pandas as pd
import numpy as np

GLOBI = "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_globi_conus_broad.csv"
OUT = "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_globi_existence_matrix.csv"

BIN_SIZE = 0.5

df = pd.read_csv(GLOBI)
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude', 
                        'sourceTaxonName', 'targetTaxonName'])

# Bin coordinates
df['lat_bin'] = (df['decimalLatitude'] // BIN_SIZE) * BIN_SIZE
df['lon_bin'] = (df['decimalLongitude'] // BIN_SIZE) * BIN_SIZE
df['bin'] = df['lat_bin'].astype(str) + "_" + df['lon_bin'].astype(str)

# Collect all species from both sides
source = df[['sourceTaxonName', 'bin']].rename(columns={'sourceTaxonName': 'species'})
target = df[['targetTaxonName', 'bin']].rename(columns={'targetTaxonName': 'species'})
all_occurrences = pd.concat([source, target]).drop_duplicates()

# Build existence matrix: species × bin
existence = all_occurrences.assign(presence=1).pivot_table(
    index='species', columns='bin', values='presence', fill_value=0
)

existence.to_csv(OUT)
print(f"Matrix shape: {existence.shape}  (species × bins)")
print(f"Sparsity: {1 - existence.values.mean():.3f}")

Matrix shape: (29720, 3268)  (species × bins)
Sparsity: 0.998


In [8]:
VALID_POLLINATOR_ORDERS = {
    'Hymenoptera', 'Lepidoptera', 'Diptera', 'Coleoptera', 'Apodiformes'
}

# Source is pollinator
source_pol = df[df['interactionTypeName'].isin(
    {'pollinates', 'visitsFlowersOf', 'visits', 'hasFlowerVisitor'}
)][['sourceTaxonName', 'sourceTaxonOrderName']]

# Target is pollinator  
target_pol = df[df['interactionTypeName'].isin(
    {'pollinatedBy', 'flowersVisitedBy', 'visitedBy'}
)][['targetTaxonName', 'targetTaxonOrderName']].rename(columns={
    'targetTaxonName': 'sourceTaxonName',
    'targetTaxonOrderName': 'sourceTaxonOrderName'
})

pollinator_species = pd.concat([source_pol, target_pol]).drop_duplicates()
pollinator_species = pollinator_species[
    pollinator_species['sourceTaxonOrderName'].isin(VALID_POLLINATOR_ORDERS)
]['sourceTaxonName'].unique()

# Plant species = everything on the other side
plant_source = df[df['interactionTypeName'].isin(
    {'pollinatedBy', 'flowersVisitedBy', 'visitedBy'}
)]['sourceTaxonName'].unique()

plant_target = df[df['interactionTypeName'].isin(
    {'pollinates', 'visitsFlowersOf', 'visits', 'hasFlowerVisitor'}
)]['targetTaxonName'].unique()

plant_species = np.union1d(plant_source, plant_target)

# Split existence matrix
F = existence[existence.index.isin(plant_species)]
P = existence[existence.index.isin(pollinator_species)]

print(f"F (plants):     {F.shape}")
print(f"P (pollinators): {P.shape}")

F.to_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence.csv")
P.to_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence.csv")

F (plants):     (20192, 3268)
P (pollinators): (7257, 3268)


In [9]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

F = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence.csv", index_col=0)
P = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence.csv", index_col=0)

# Fit PCA separately on each matrix
pca_f = PCA(n_components=15, random_state=42)
pca_p = PCA(n_components=15, random_state=42)

Vf = pca_f.fit_transform(F.values)
Vp = pca_p.fit_transform(P.values)

# Wrap back into dataframes with species names as index
Vf_df = pd.DataFrame(Vf, index=F.index, columns=[f'f_pc{i+1}' for i in range(15)])
Vp_df = pd.DataFrame(Vp, index=P.index, columns=[f'p_pc{i+1}' for i in range(15)])

Vf_df.to_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vf.csv")
Vp_df.to_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vp.csv")

print(f"Vf shape: {Vf_df.shape}")
print(f"Vp shape: {Vp_df.shape}")
print(f"\nF variance explained by 15 PCs: {pca_f.explained_variance_ratio_.sum():.3f}")
print(f"P variance explained by 15 PCs: {pca_p.explained_variance_ratio_.sum():.3f}")

Vf shape: (20192, 15)
Vp shape: (7257, 15)

F variance explained by 15 PCs: 0.339
P variance explained by 15 PCs: 0.407


In [10]:
import os
files = [
    "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007204_observations_v2.csv",
    "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv"
]
for f in files:
    print(f"{os.path.basename(f)}: {os.path.exists(f)}")

gbif_0007204_observations_v2.csv: True
gbif_0007192_observations_v2.csv: True


In [ ]:
import pandas as pd

GBIF_PATH_2 = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv"
OUT = "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence_gbif_combined.csv"

BIN_SIZE = 0.5
CONUS_LON = (-125, -66)
CONUS_LAT  = (24, 50)
CHUNKSIZE  = 500_000

all_pairs = []

for i, chunk in enumerate(pd.read_csv(GBIF_PATH_2, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = chunk.dropna(subset=['lat', 'lon', 'pollinator_species'])
    chunk = chunk[
        chunk['lon'].between(*CONUS_LON) &
        chunk['lat'].between(*CONUS_LAT)
    ]
    chunk['bin'] = (
        ((chunk['lat'] // BIN_SIZE) * BIN_SIZE).astype(str) + "_" +
        ((chunk['lon'] // BIN_SIZE) * BIN_SIZE).astype(str)
    )
    pairs = chunk[['pollinator_species', 'bin']].drop_duplicates()
    all_pairs.append(pairs)

    if i % 2 == 0:
        print(f"  Processed {(i+1)*CHUNKSIZE/1e6:.0f}M rows...")

# Merge with the 0007204 result
print("Merging with existing P matrix...")
new_pairs = pd.concat(all_pairs).drop_duplicates()

P_bird = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence_gbif.csv", index_col=0)

# Convert bird matrix back to pairs
bird_pairs = P_bird.stack().reset_index()
bird_pairs.columns = ['pollinator_species', 'bin', 'presence']
bird_pairs = bird_pairs[bird_pairs['presence'] == 1][['pollinator_species', 'bin']]

# Combine all pairs and pivot once
all_combined = pd.concat([new_pairs, bird_pairs]).drop_duplicates()
all_combined['presence'] = 1

P_combined = all_combined.pivot_table(
    index='pollinator_species', columns='bin',
    values='presence', fill_value=0
)

P_combined.to_csv(OUT)
print(f"Combined P matrix shape: {P_combined.shape}")
print(f"Sparsity: {1 - P_combined.values.mean():.3f}")

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

P_combined = pd.read_csv(
    "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence_gbif_combined.csv", 
    index_col=0
)

pca_p = PCA(n_components=15, random_state=42)
Vp = pca_p.fit_transform(P_combined.values)

Vp_df = pd.DataFrame(Vp, index=P_combined.index, 
                      columns=[f'p_pc{i+1}' for i in range(15)])

Vp_df.to_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vp_gbif.csv")

print(f"Vp shape: {Vp_df.shape}")
print(f"P variance explained by 15 PCs: {pca_p.explained_variance_ratio_.sum():.3f}")

In [ ]:
F = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence.csv", index_col=0)

# 每个植物物种出现在多少个bin里
bin_counts = F.sum(axis=1)

print(f"Total plant species: {len(bin_counts):,}")
print(f"\nBin count distribution:")
print(f"  median: {bin_counts.median():.0f} bins")
print(f"  mean:   {bin_counts.mean():.1f} bins")
print(f"  <5 bins: {(bin_counts < 5).sum():,} species ({(bin_counts < 5).mean()*100:.1f}%)")
print(f"  <2 bins: {(bin_counts < 2).sum():,} species ({(bin_counts < 2).mean()*100:.1f}%)")

In [ ]:
import os
import subprocess

# 看看rawplantsdata里现在有什么
result = subprocess.run(['ls', '-lh', '/scratch/ariana.l/CfE2026CVforEcology/rawplantsdata/'], 
                       capture_output=True, text=True)
print(result.stdout)

In [ ]:
result = subprocess.run(['ls', '/scratch/ariana.l/hf_cache/'], 
                       capture_output=True, text=True)
print(result.stdout)

In [ ]:
result = subprocess.run(['ls', '/scratch/ariana.l/hf_cache/datasets/'], 
                       capture_output=True, text=True)
print(result.stdout)

In [ ]:
import pandas as pd

fc = pd.read_parquet("/scratch/ariana.l/ppe-outputs/data/flowering_curves_used.parquet")
print(fc.shape)
print(fc.columns.tolist())
print(fc.head(3))

In [ ]:
subprocess.run(['pip', 'install', 'datasets', 'huggingface_hub'])

In [ ]:
import os
os.environ["HF_HOME"] = "/scratch/ariana.l/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/scratch/ariana.l/hf_cache/datasets"

import io, time
import pandas as pd
import pyarrow.parquet as pq
from datasets import Dataset
from huggingface_hub import hf_hub_download
from pathlib import Path

REPO_ID = "dcher95/phenofield"
NUM_SHARDS = 107
OUT = "/scratch/ariana.l/Stage 4 Link Prediction Model/phenofield_plant_occurrences.parquet"

all_dfs = []
start = time.time()

for i in range(NUM_SHARDS):
    path = hf_hub_download(repo_id=REPO_ID, 
                           filename=f"train/data-{i:05d}-of-00107.arrow",
                           repo_type="dataset")
    table = Dataset.from_file(path).data.table.select(["species", "latitude", "longitude"])
    df = table.to_pandas().rename(columns={"latitude": "lat", "longitude": "lon"})
    all_dfs.append(df)
    
    if (i+1) % 10 == 0:
        elapsed = time.time() - start
        print(f"Shard {i+1}/{NUM_SHARDS} | elapsed: {elapsed:.0f}s")

result = pd.concat(all_dfs, ignore_index=True)
result.to_parquet(OUT, index=False)
print(f"Done! Shape: {result.shape}")
print(f"Unique species: {result['species'].nunique()}")

In [ ]:
import pandas as pd

BIN_SIZE = 0.5
CONUS_LON = (-125, -66)
CONUS_LAT = (24, 50)
OUT = "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence_phenofield.csv"

plants = pd.read_parquet("/scratch/ariana.l/Stage 4 Link Prediction Model/phenofield_plant_occurrences.parquet")

# CONUS filter
plants = plants.dropna(subset=['lat', 'lon'])
plants = plants[
    plants['lon'].between(*CONUS_LON) &
    plants['lat'].between(*CONUS_LAT)
]

# Bin
plants['bin'] = (
    ((plants['lat'] // BIN_SIZE) * BIN_SIZE).astype(str) + "_" +
    ((plants['lon'] // BIN_SIZE) * BIN_SIZE).astype(str)
)

# Pivot to existence matrix
plants['presence'] = 1
F = plants[['species', 'bin', 'presence']].drop_duplicates().pivot_table(
    index='species', columns='bin', values='presence', fill_value=0
)

F.to_csv(OUT)
print(f"F matrix shape: {F.shape}")
print(f"Sparsity: {1 - F.values.mean():.3f}")

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

F = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence_phenofield.csv", index_col=0)

pca_f = PCA(n_components=15, random_state=42)
Vf = pca_f.fit_transform(F.values)

Vf_df = pd.DataFrame(Vf, index=F.index, columns=[f'f_pc{i+1}' for i in range(15)])
Vf_df.to_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vf_phenofield.csv")

print(f"Vf shape: {Vf_df.shape}")
print(f"F variance explained by 15 PCs: {pca_f.explained_variance_ratio_.sum():.3f}")

In [ ]:
import pandas as pd

Vf_df = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vf_phenofield.csv", index_col=0)
Vp_df = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vp_gbif.csv", index_col=0)
F = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence_gbif_combined.csv", index_col=0)
globi = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_globi_conus_broad.csv")

# Get common bins between F and P matrices
common_bins = F.columns.intersection(P.columns)
F_common = F[common_bins]
P_common = P[common_bins]

print(f"Common bins between F and P: {len(common_bins)}")
print(f"F species in Vf: {len(Vf_df)}")
print(f"P species in Vp: {len(Vp_df)}")

In [ ]:
import numpy as np

# Get GloBI interaction pairs
source_is_pollinator = {'visitsFlowersOf', 'pollinates', 'visits', 'hasFlowerVisitor'}
target_is_pollinator = {'flowersVisitedBy', 'pollinatedBy', 'visitedBy'}

pairs_source = globi[globi['interactionTypeName'].isin(source_is_pollinator)][
    ['targetTaxonName', 'sourceTaxonName']].rename(
    columns={'targetTaxonName': 'plant', 'sourceTaxonName': 'pollinator'})

pairs_target = globi[globi['interactionTypeName'].isin(target_is_pollinator)][
    ['sourceTaxonName', 'targetTaxonName']].rename(
    columns={'sourceTaxonName': 'plant', 'targetTaxonName': 'pollinator'})

pairs = pd.concat([pairs_source, pairs_target]).drop_duplicates()

# Filter to species that exist in both Vf and Vp
pairs = pairs[
    pairs['plant'].isin(Vf_df.index) &
    pairs['pollinator'].isin(Vp_df.index)
]

print(f"Total GloBI pairs: {len(pairs):,}")

# Compute N = dot product of binary vectors over common bins
F_arr = F_common.values  # (6466, 3160)
P_arr = P_common.values  # (4515, 3160)

# Map species to row index
f_idx = {s: i for i, s in enumerate(F_common.index)}
p_idx = {s: i for i, s in enumerate(P_common.index)}

pairs['N'] = pairs.apply(
    lambda r: int(F_arr[f_idx[r['plant']]] @ P_arr[p_idx[r['pollinator']]]),
    axis=1
)

print(f"N statistics:")
print(pairs['N'].describe())

In [ ]:
import numpy as np

Vf_arr = Vf_df.values  # (6466, 15)
Vp_arr = Vp_df.values  # (4515, 15)

vf_idx = {s: i for i, s in enumerate(Vf_df.index)}
vp_idx = {s: i for i, s in enumerate(Vp_df.index)}

# Build feature matrix: [Vf (15) + Vp (15) + N (1)] = 31D
X = np.hstack([
    np.array([Vf_arr[vf_idx[r['plant']]] for _, r in pairs.iterrows()]),
    np.array([Vp_arr[vp_idx[r['pollinator']]] for _, r in pairs.iterrows()]),
    pairs['N'].values.reshape(-1, 1)
])

y = np.ones(len(pairs), dtype=int)  # all GloBI pairs = positive label (1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"All labels: {np.unique(y)}")

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

np.random.seed(42)

# All possible (plant, pollinator) pairs
all_plants = list(Vf_df.index)
all_pollinators = list(Vp_df.index)

# Existing positive pairs as a set
positive_set = set(zip(pairs['plant'], pairs['pollinator']))

# Sample negatives
n_neg = len(pairs) * 3  # ~25% positive ratio
neg_pairs = []
while len(neg_pairs) < n_neg:
    p = np.random.choice(all_plants)
    pol = np.random.choice(all_pollinators)
    if (p, pol) not in positive_set:
        neg_pairs.append((p, pol))

neg_df = pd.DataFrame(neg_pairs, columns=['plant', 'pollinator'])

# Compute N for negatives
neg_df['N'] = neg_df.apply(
    lambda r: int(F_arr[f_idx[r['plant']]] @ P_arr[p_idx[r['pollinator']]]),
    axis=1
)

# Build negative feature matrix
X_neg = np.hstack([
    np.array([Vf_arr[vf_idx[r['plant']]] for _, r in neg_df.iterrows()]),
    np.array([Vp_arr[vp_idx[r['pollinator']]] for _, r in neg_df.iterrows()]),
    neg_df['N'].values.reshape(-1, 1)
])
y_neg = np.zeros(len(neg_df), dtype=int)

# Combine
X_all = np.vstack([X, X_neg])
y_all = np.concatenate([y, y_neg])

print(f"Total pairs: {len(y_all):,} ({y.sum():,} positive, {y_neg.sum()==0 and len(y_neg):,} negative)")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Logistic regression
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
print(f"\nBaseline A2 (no PPE):")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_pred_proba):.3f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_pred_proba):.3f}")

In [ ]:
import pickle

# Save model
with open("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_A2_logistic.pkl", "wb") as f:
    pickle.dump(clf, f)

# Save results
results_A2 = {
    'model': 'A2_baseline',
    'features': '[Vf, Vp, N]',
    'n_positive': len(pairs),
    'n_negative': len(neg_df),
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'pr_auc': average_precision_score(y_test, y_pred_proba)
}

print("A2 saved. Ready for PPE integration (A3) once opportunity_surface transfer completes.")

In [ ]:
import pandas as pd
import glob

files = sorted(glob.glob("/scratch/ariana.l/ppe-outputs/opportunity_surface/part_*.parquet"))
print(f"Files available: {len(files)}")

# Peek at one file
sample = pd.read_parquet(files[0])
print(f"\nShape: {sample.shape}")
print(f"Columns: {sample.columns.tolist()}")
print(sample.head(3))

In [ ]:
import pandas as pd
import glob

files = sorted(glob.glob("/scratch/ariana.l/ppe-outputs/opportunity_surface/part_*.parquet"))

# Check species coverage
species_set = set()
for f in files:
    df = pd.read_parquet(f, columns=['species'])
    species_set.update(df['species'].unique())

print(f"Plant species covered by 1000 PPE files: {len(species_set):,}")

# How many overlap with our GloBI pairs?
globi_plants = set(pairs['plant'].unique())
overlap = species_set & globi_plants
print(f"GloBI plant species in pairs: {len(globi_plants):,}")
print(f"Covered by PPE so far: {len(overlap):,} ({len(overlap)/len(globi_plants)*100:.1f}%)")

In [ ]:
#checking coverage
import pandas as pd
import glob

files = sorted(glob.glob("/scratch/ariana.l/ppe-outputs/opportunity_surface/part_*.parquet"))
print(f"Files available: {len(files)}")

species_set = set()
for f in files:
    df = pd.read_parquet(f, columns=['species'])
    species_set.update(df['species'].unique())

globi_plants = set(pairs['plant'].unique())
overlap = species_set & globi_plants
print(f"Plant species in PPE: {len(species_set):,}")
print(f"GloBI plant species in pairs: {len(globi_plants):,}")
print(f"Covered by PPE: {len(overlap):,} ({len(overlap)/len(globi_plants)*100:.1f}%)")

In [ ]:
#loading all the PPE flowering curves
import pandas as pd
import glob
from tqdm import tqdm

files = sorted(glob.glob("/scratch/ariana.l/ppe-outputs/opportunity_surface/part_*.parquet"))

# Load flowering curves for all species
# We want: species -> array of 52 weekly norm values (averaged across all cells)
all_curves = []
for f in files:
    df = pd.read_parquet(f, columns=['species', 'week', 'norm'])
    # Average norm across all cells per species per week
    curve = df.groupby(['species', 'week'])['norm'].mean().reset_index()
    all_curves.append(curve)

flowering_curves = pd.concat(all_curves).groupby(['species', 'week'])['norm'].mean().reset_index()
flowering_curves = flowering_curves.pivot(index='species', columns='week', values='norm').fillna(0)

print(f"Flowering curves shape: {flowering_curves.shape}")
print(f"Sample species: {flowering_curves.index[:3].tolist()}")

In [ ]:
import numpy as np
from scipy.stats import vonmises

GBIF_PATH = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv"
WEEKS = 52

def doy_to_week(doy):
    return ((doy - 1) // 7).clip(0, 51)

# Load pollinator doy observations
pol_doy = pd.read_csv(GBIF_PATH, usecols=['pollinator_species', 'doy'], low_memory=False)
pol_doy = pol_doy.dropna(subset=['doy', 'pollinator_species'])
pol_doy['week'] = doy_to_week(pol_doy['doy'].astype(int))

# Build activity curve per pollinator species
activity_curves = pol_doy.groupby(['pollinator_species', 'week']).size().reset_index(name='count')
activity_curves = activity_curves.pivot(index='pollinator_species', columns='week', values='count').fillna(0)

# Normalize to sum to 1
activity_curves = activity_curves.div(activity_curves.sum(axis=1), axis=0)

print(f"Activity curves shape: {activity_curves.shape}")
print(f"Pollinators with activity data: {len(activity_curves):,}")

# Check overlap with our pairs
pol_in_pairs = set(pairs['pollinator'].unique())
pol_covered = set(activity_curves.index) & pol_in_pairs
print(f"Pollinators in pairs covered: {len(pol_covered):,} / {len(pol_in_pairs):,} ({len(pol_covered)/len(pol_in_pairs)*100:.1f}%)")

In [ ]:
import numpy as np

# Align to 52 weeks
weeks = list(range(52))
f_curves = flowering_curves.reindex(columns=weeks, fill_value=0)
a_curves = activity_curves.reindex(columns=weeks, fill_value=0)

# Compute Delta for each pair
# Delta = sum_t min(f_P(t), a_A(t)), normalized
def compute_delta(plant, pollinator):
    f = f_curves.loc[plant].values
    a = a_curves.loc[pollinator].values
    return np.minimum(f, a).sum()

# Filter pairs to those with both flowering and activity curves
pairs_a3 = pairs[
    pairs['plant'].isin(f_curves.index) &
    pairs['pollinator'].isin(a_curves.index)
].copy()

print(f"Pairs for A3: {len(pairs_a3):,} (out of {len(pairs):,})")

pairs_a3['delta'] = pairs_a3.apply(
    lambda r: compute_delta(r['plant'], r['pollinator']), axis=1
)

print(f"\nDelta statistics:")
print(pairs_a3['delta'].describe())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pickle

# Build 32D feature matrix for A3 pairs
X_a3 = np.hstack([
    np.array([Vf_arr[vf_idx[r['plant']]] for _, r in pairs_a3.iterrows()]),
    np.array([Vp_arr[vp_idx[r['pollinator']]] for _, r in pairs_a3.iterrows()]),
    pairs_a3['N'].values.reshape(-1, 1),
    pairs_a3['delta'].values.reshape(-1, 1)
])
y_a3 = np.ones(len(pairs_a3), dtype=int)

print(f"X_a3 shape: {X_a3.shape}")  # should be (3074, 32)

# Sample negatives for A3 (same ratio)
np.random.seed(42)
positive_set_a3 = set(zip(pairs_a3['plant'], pairs_a3['pollinator']))
n_neg = len(pairs_a3) * 3
neg_pairs_a3 = []
while len(neg_pairs_a3) < n_neg:
    p = np.random.choice(list(f_curves.index))
    pol = np.random.choice(list(a_curves.index))
    if (p, pol) not in positive_set_a3 and pol in vp_idx and p in vf_idx:
        neg_pairs_a3.append((p, pol))

neg_df_a3 = pd.DataFrame(neg_pairs_a3, columns=['plant', 'pollinator'])
neg_df_a3['N'] = neg_df_a3.apply(
    lambda r: int(F_arr[f_idx[r['plant']]] @ P_arr[p_idx[r['pollinator']]]) 
    if r['plant'] in f_idx and r['pollinator'] in p_idx else 0, axis=1
)
neg_df_a3['delta'] = neg_df_a3.apply(
    lambda r: compute_delta(r['plant'], r['pollinator']), axis=1
)

X_neg_a3 = np.hstack([
    np.array([Vf_arr[vf_idx[r['plant']]] for _, r in neg_df_a3.iterrows()]),
    np.array([Vp_arr[vp_idx[r['pollinator']]] for _, r in neg_df_a3.iterrows()]),
    neg_df_a3['N'].values.reshape(-1, 1),
    neg_df_a3['delta'].values.reshape(-1, 1)
])
y_neg_a3 = np.zeros(len(neg_df_a3), dtype=int)

X_all_a3 = np.vstack([X_a3, X_neg_a3])
y_all_a3 = np.concatenate([y_a3, y_neg_a3])

X_train, X_test, y_train, y_test = train_test_split(
    X_all_a3, y_all_a3, test_size=0.2, random_state=42, stratify=y_all_a3)

clf_a3 = LogisticRegression(max_iter=1000, random_state=42)
clf_a3.fit(X_train, y_train)

y_pred_a3 = clf_a3.predict_proba(X_test)[:, 1]
print(f"\nA3 (with PPE Delta):")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_a3):.3f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_pred_a3):.3f}")
print(f"\nA2 baseline:")
print(f"ROC-AUC: 0.931")
print(f"PR-AUC:  0.842")

with open("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_A3_logistic.pkl", "wb") as f:
    pickle.dump(clf_a3, f)

In [ ]:
import numpy as np
import pandas as pd

# Rebuild A2 predictions on the same A3 test set for fair comparison
# First we need A2 features for the A3 pairs (no delta)
X_a2_subset = X_all_a3[:, :31]  # drop the delta column

X_train_a2, X_test_a2, y_train_a2, y_test_a2 = train_test_split(
    X_a2_subset, y_all_a3, test_size=0.2, random_state=42, stratify=y_all_a3)

clf_a2_subset = LogisticRegression(max_iter=1000, random_state=42)
clf_a2_subset.fit(X_train_a2, y_train_a2)
y_pred_a2_subset = clf_a2_subset.predict_proba(X_test_a2)[:, 1]

# Get test set indices
test_idx = np.where(np.isin(np.arange(len(y_all_a3)), 
                             train_test_split(np.arange(len(y_all_a3)), 
                             test_size=0.2, random_state=42, 
                             stratify=y_all_a3)[1]))[0]

# Build test dataframe
all_pairs_a3 = pd.concat([
    pairs_a3.assign(label=1),
    neg_df_a3.assign(label=0)
]).reset_index(drop=True)

test_df = all_pairs_a3.iloc[test_idx].copy()
test_df['y_true'] = y_test_a2
test_df['pred_a2'] = y_pred_a2_subset
test_df['pred_a3'] = y_pred_a3

# Add bin counts as proxy for species popularity
plant_bin_counts = F.sum(axis=1).rename('plant_bin_count')
pol_bin_counts = P.sum(axis=1).rename('pol_bin_count')

test_df = test_df.join(plant_bin_counts, on='plant')
test_df = test_df.join(pol_bin_counts, on='pollinator')

# Bin by quartile
test_df['plant_popularity'] = pd.qcut(test_df['plant_bin_count'], 4, 
                                       labels=['rare','low','mid','common'])
test_df['pol_popularity'] = pd.qcut(test_df['pol_bin_count'], 4,
                                     labels=['rare','low','mid','common'])

print(test_df[['plant_popularity', 'pol_popularity', 
               'pred_a2', 'pred_a3', 'y_true']].head())

In [ ]:
from sklearn.metrics import roc_auc_score

# Only look at positive pairs for this analysis
positives = test_df[test_df['y_true'] == 1].copy()

print("=== Improvement by PLANT popularity ===")
for group in ['rare', 'low', 'mid', 'common']:
    subset = test_df[test_df['plant_popularity'] == group]
    if subset['y_true'].sum() < 5:
        continue
    try:
        auc_a2 = roc_auc_score(subset['y_true'], subset['pred_a2'])
        auc_a3 = roc_auc_score(subset['y_true'], subset['pred_a3'])
        print(f"  {group:8s}: A2={auc_a2:.3f}  A3={auc_a3:.3f}  diff={auc_a3-auc_a2:+.3f}")
    except:
        print(f"  {group:8s}: insufficient data")

print("\n=== Improvement by POLLINATOR popularity ===")
for group in ['rare', 'low', 'mid', 'common']:
    subset = test_df[test_df['pol_popularity'] == group]
    if subset['y_true'].sum() < 5:
        continue
    try:
        auc_a2 = roc_auc_score(subset['y_true'], subset['pred_a2'])
        auc_a3 = roc_auc_score(subset['y_true'], subset['pred_a3'])
        print(f"  {group:8s}: A2={auc_a2:.3f}  A3={auc_a3:.3f}  diff={auc_a3-auc_a2:+.3f}")
    except:
        print(f"  {group:8s}: insufficient data")

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("=== Improvement by PLANT popularity ===")
print(f"  {'group':8s}  {'ROC A2':>8} {'ROC A3':>8} {'ROC diff':>9}  {'PR A2':>8} {'PR A3':>8} {'PR diff':>9}")
for group in ['rare', 'low', 'mid', 'common']:
    subset = test_df[test_df['plant_popularity'] == group]
    if subset['y_true'].sum() < 5:
        continue
    try:
        roc_a2 = roc_auc_score(subset['y_true'], subset['pred_a2'])
        roc_a3 = roc_auc_score(subset['y_true'], subset['pred_a3'])
        pr_a2  = average_precision_score(subset['y_true'], subset['pred_a2'])
        pr_a3  = average_precision_score(subset['y_true'], subset['pred_a3'])
        print(f"  {group:8s}  {roc_a2:8.3f} {roc_a3:8.3f} {roc_a3-roc_a2:+9.3f}  {pr_a2:8.3f} {pr_a3:8.3f} {pr_a3-pr_a2:+9.3f}")
    except:
        print(f"  {group:8s}: insufficient data")

print("\n=== Improvement by POLLINATOR popularity ===")
print(f"  {'group':8s}  {'ROC A2':>8} {'ROC A3':>8} {'ROC diff':>9}  {'PR A2':>8} {'PR A3':>8} {'PR diff':>9}")
for group in ['rare', 'low', 'mid', 'common']:
    subset = test_df[test_df['pol_popularity'] == group]
    if subset['y_true'].sum() < 5:
        continue
    try:
        roc_a2 = roc_auc_score(subset['y_true'], subset['pred_a2'])
        roc_a3 = roc_auc_score(subset['y_true'], subset['pred_a3'])
        pr_a2  = average_precision_score(subset['y_true'], subset['pred_a2'])
        pr_a3  = average_precision_score(subset['y_true'], subset['pred_a3'])
        print(f"  {group:8s}  {roc_a2:8.3f} {roc_a3:8.3f} {roc_a3-roc_a2:+9.3f}  {pr_a2:8.3f} {pr_a3:8.3f} {pr_a3-pr_a2:+9.3f}")
    except:
        print(f"  {group:8s}: insufficient data")

In [ ]:
globi = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_globi_conus_broad.csv")

# Count observations per pair
source_is_pollinator = {'visitsFlowersOf', 'pollinates', 'visits', 'hasFlowerVisitor'}
target_is_pollinator = {'flowersVisitedBy', 'pollinatedBy', 'visitedBy'}

obs_source = globi[globi['interactionTypeName'].isin(source_is_pollinator)][
    ['targetTaxonName', 'sourceTaxonName', 'decimalLatitude', 'decimalLongitude']].rename(
    columns={'targetTaxonName': 'plant', 'sourceTaxonName': 'pollinator'})

obs_target = globi[globi['interactionTypeName'].isin(target_is_pollinator)][
    ['sourceTaxonName', 'targetTaxonName', 'decimalLatitude', 'decimalLongitude']].rename(
    columns={'sourceTaxonName': 'plant', 'targetTaxonName': 'pollinator'})

obs_all = pd.concat([obs_source, obs_target]).dropna(subset=['decimalLatitude', 'decimalLongitude'])

pair_counts = obs_all.groupby(['plant', 'pollinator']).size().reset_index(name='n_obs')
pair_counts = pair_counts[
    pair_counts['plant'].isin(pairs_a3['plant'].values) &
    pair_counts['pollinator'].isin(pairs_a3['pollinator'].values)
].sort_values('n_obs', ascending=False)

print("Top 15 pairs by observation count:")
print(pair_counts.head(15).to_string())

In [ ]:
pol_orders = globi[['sourceTaxonName', 'sourceTaxonOrderName']].rename(
    columns={'sourceTaxonName': 'pollinator', 'sourceTaxonOrderName': 'order'})

top_pairs_with_order = pair_counts.head(15).merge(pol_orders.drop_duplicates(), on='pollinator', how='left')
print(top_pairs_with_order[['plant', 'pollinator', 'n_obs', 'order']].to_string())

In [ ]:
# Find best pairs from our original 5 pollinator orders
valid_orders = {'Hymenoptera', 'Lepidoptera', 'Diptera', 'Coleoptera', 'Apodiformes'}

pol_order_map = globi[['sourceTaxonName', 'sourceTaxonOrderName']].rename(
    columns={'sourceTaxonName': 'pollinator', 'sourceTaxonOrderName': 'order'}
).drop_duplicates()

pair_counts_filtered = pair_counts.merge(pol_order_map, on='pollinator', how='left')
pair_counts_filtered = pair_counts_filtered[
    pair_counts_filtered['order'].isin(valid_orders)
].sort_values('n_obs', ascending=False)

print("Top 15 pairs from valid pollinator orders:")
print(pair_counts_filtered.head(15)[['plant', 'pollinator', 'n_obs', 'order']].to_string())

In [ ]:
order_map_source = globi[['sourceTaxonName', 'sourceTaxonOrderName']].rename(
    columns={'sourceTaxonName': 'pollinator', 'sourceTaxonOrderName': 'order'})
order_map_target = globi[['targetTaxonName', 'targetTaxonOrderName']].rename(
    columns={'targetTaxonName': 'pollinator', 'targetTaxonOrderName': 'order'})

pol_order_map = pd.concat([order_map_source, order_map_target]).drop_duplicates(subset='pollinator')

pair_counts_filtered = pair_counts.merge(pol_order_map, on='pollinator', how='left')
pair_counts_filtered = pair_counts_filtered[
    pair_counts_filtered['order'].isin(valid_orders)
].sort_values('n_obs', ascending=False)

print(f"Total pairs with valid orders: {len(pair_counts_filtered):,}")
print(pair_counts_filtered.head(15)[['plant', 'pollinator', 'n_obs', 'order']].to_string())

In [ ]:
# Check what pollinators are in pair_counts
print("Sample pollinators in pair_counts:")
print(pair_counts['pollinator'].head(10).tolist())

# Check what's in pol_order_map for valid orders
valid_in_map = pol_order_map[pol_order_map['order'].isin(valid_orders)]
print(f"\nValid order species in order map: {len(valid_in_map):,}")
print(valid_in_map.head(5).to_string())

# Check overlap
overlap = set(pair_counts['pollinator']) & set(valid_in_map['pollinator'])
print(f"\nOverlap: {len(overlap):,}")

In [ ]:
# pairs_a3 already has correctly oriented plant/pollinator
# Just count how many GloBI records each pair has

obs_all['bin'] = (
    ((obs_all['decimalLatitude'] // 0.5) * 0.5).astype(str) + "_" +
    ((obs_all['decimalLongitude'] // 0.5) * 0.5).astype(str)
)

pair_obs_counts = obs_all.groupby(['plant', 'pollinator']).agg(
    n_obs=('decimalLatitude', 'count'),
    n_bins=('bin', 'nunique')
).reset_index()

# Filter to pairs in pairs_a3
valid_pairs = pairs_a3[['plant', 'pollinator']].merge(pair_obs_counts, on=['plant', 'pollinator'], how='left')
valid_pairs = valid_pairs.dropna().sort_values('n_obs', ascending=False)

# Add pollinator order
valid_pairs = valid_pairs.merge(pol_order_map, on='pollinator', how='left')
valid_pairs = valid_pairs[valid_pairs['order'].isin(valid_orders)]

print(valid_pairs.head(15)[['plant', 'pollinator', 'n_obs', 'n_bins', 'order']].to_string())

In [ ]:
# Direct approach: look at pairs_a3 pollinators and check their order in GBIF data
gbif_path = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv"
gbif_sample = pd.read_csv(gbif_path, usecols=['pollinator_species'], nrows=100000)

# What pollinators from pairs_a3 appear in the insect GBIF file?
gbif_pollinators = set(gbif_sample['pollinator_species'].dropna().unique())
pairs_a3_pollinators = set(pairs_a3['pollinator'].unique())

overlap = pairs_a3_pollinators & gbif_pollinators
print(f"pairs_a3 pollinators found in insect GBIF: {len(overlap):,}")
print("\nSample:")
print(list(overlap)[:10])

In [ ]:
target_pairs = [
    ('Asclepias syriaca', 'Danaus plexippus'),
    ('Asclepias tuberosa', 'Danaus plexippus'),
    ('Echinacea purpurea', 'Danaus plexippus'),
]

for plant, pollinator in target_pairs:
    in_a3 = len(pairs_a3[(pairs_a3['plant']==plant) & (pairs_a3['pollinator']==pollinator)]) > 0
    n = len(obs_all[(obs_all['plant']==plant) & (obs_all['pollinator']==pollinator)])
    print(f"{plant} / {pollinator}: in_pairs_a3={in_a3}, n_obs={n}")

In [ ]:
print(f"Danaus plexippus in Vp: {'Danaus plexippus' in Vp_df.index}")
print(f"Asclepias syriaca in Vf: {'Asclepias syriaca' in Vf_df.index}")
print(f"Asclepias syriaca in flowering_curves: {'Asclepias syriaca' in flowering_curves.index}")
print(f"Danaus plexippus in activity_curves: {'Danaus plexippus' in activity_curves.index}")

In [ ]:
# Load full gbif_0007192 species list
gbif_full = pd.read_csv(
    "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv",
    usecols=['pollinator_species'], low_memory=False
)
gbif_insect_species = set(gbif_full['pollinator_species'].dropna().unique())

# Find pairs_a3 pollinators that are in insect GBIF
pairs_a3_insect = pairs_a3[pairs_a3['pollinator'].isin(gbif_insect_species)]
print(f"pairs_a3 with insect GBIF pollinators: {len(pairs_a3_insect):,}")

# Count GloBI obs for these pairs
pairs_a3_insect_obs = pairs_a3_insect.merge(pair_obs_counts, on=['plant','pollinator'], how='left')
pairs_a3_insect_obs = pairs_a3_insect_obs.dropna().sort_values('n_obs', ascending=False)
print(pairs_a3_insect_obs.head(10)[['plant','pollinator','n_obs']].to_string())

In [ ]:
plant = 'Achillea millefolium'
print(f"In Vf: {plant in Vf_df.index}")
print(f"In flowering_curves: {plant in flowering_curves.index}")

# How many GloBI pairs with this plant?
plant_pairs = pairs_a3[pairs_a3['plant'] == plant]
print(f"Pairs in A3: {len(plant_pairs)}")
print(plant_pairs[['pollinator']].head(10).to_string())

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("/scratch/ariana.l/ppe-outputs/opportunity_surface/part_*.parquet"))

# Find which file has Achillea millefolium
achillea_data = []
for f in files:
    df = pd.read_parquet(f)
    subset = df[df['species'] == 'Achillea millefolium']
    if len(subset) > 0:
        achillea_data.append(subset)

achillea = pd.concat(achillea_data)
print(f"Shape: {achillea.shape}")
print(f"Unique cells: {achillea['cell_idx'].nunique()}")
print(f"Weeks: {sorted(achillea['week'].unique())}")
print(achillea.head(3))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Month -> week mapping
MONTHS = {
    'Feb': 4,
    'Mar': 9, 
    'Apr': 13,
    'May': 18
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Achillea millefolium — PPE Flowering Probability (ANTHEIA)', 
             fontsize=14, fontweight='bold', y=1.02)

for ax, (month, week) in zip(axes, MONTHS.items()):
    week_data = achillea[achillea['week'] == week]
    
    sc = ax.scatter(
        week_data['centroid_lon'],
        week_data['centroid_lat'],
        c=week_data['p_flowering'],
        cmap='RdYlGn',
        s=3,
        vmin=0, vmax=1,
        alpha=0.8
    )
    
    ax.set_xlim(-125, -66)
    ax.set_ylim(24, 50)
    ax.set_title(f'{month}', fontsize=12)
    ax.set_xlabel('Longitude')
    if ax == axes[0]:
        ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

plt.colorbar(sc, ax=axes, label='P(flowering)', shrink=0.8)
plt.savefig('/scratch/ariana.l/Stage 4 Link Prediction Model/antheia_flowering_achillea.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

MONTHS = {'Feb': 4, 'Mar': 9, 'Apr': 13, 'May': 18}
PLANT = 'Achillea millefolium'

fig = plt.figure(figsize=(22, 12))
gs = gridspec.GridSpec(3, 5, width_ratios=[1,1,1,1,0.08], 
                       hspace=0.35, wspace=0.15)

row_titles = ['Ground Truth (GloBI)', 'Spatial Co-occurrence (A2)', 'ANTHEIA (A3 + PPE Δ)']

for row in range(3):
    for col, (month_name, week) in enumerate(MONTHS.items()):
        ax = fig.add_subplot(gs[row, col])
        week_data = achillea[achillea['week'] == week]

        if row == 0:
            ax.scatter(gt['decimalLongitude'], gt['decimalLatitude'],
                      c='darkgreen', s=8, alpha=0.6)
        elif row == 1:
            ax.scatter(week_data['centroid_lon'], week_data['centroid_lat'],
                      c=[avg_a2]*len(week_data), cmap='RdYlGn',
                      s=3, vmin=0, vmax=1, alpha=0.8)
        else:
            a3_preds = []
            for _, r in plant_pol_pairs.iterrows():
                pol = r['pollinator']
                if pol not in vp_idx or pol not in activity_curves.index: continue
                f_w = flowering_curves.loc[PLANT, week] if week in flowering_curves.columns else 0
                a_w = activity_curves.loc[pol, week] if week in activity_curves.columns else 0
                feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [r['N'], min(f_w, a_w)]])
                a3_preds.append(clf_a3.predict_proba(feat.reshape(1,-1))[0,1])
            avg_a3 = np.mean(a3_preds) if a3_preds else 0
            sc = ax.scatter(week_data['centroid_lon'], week_data['centroid_lat'],
                      c=[avg_a3]*len(week_data), cmap='RdYlGn',
                      s=3, vmin=0, vmax=1, alpha=0.8)

        ax.set_xlim(-125, -66); ax.set_ylim(24, 50); ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])

        # Column title (month) on top row only
        if row == 0:
            ax.set_title(month_name, fontsize=12, fontweight='bold', pad=8)

        # Row title centered above first column of each row
        if col == 0:
            ax.text(-0.08, 1.12, row_titles[row], transform=ax.transAxes,
                   fontsize=11, fontweight='bold', ha='left', va='bottom')

# Colorbar in last column
cax = fig.add_subplot(gs[1:, 4])
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(0, 1))
sm.set_array([])
plt.colorbar(sm, cax=cax, label='P(interaction)')

fig.suptitle('Achillea millefolium — Interaction Predictions vs Ground Truth',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('/scratch/ariana.l/Stage 4 Link Prediction Model/antheia_comparison_achillea_v2.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

In [ ]:
import numpy as np
import pandas as pd

PLANT = 'Achillea millefolium'
BIN_SIZE = 0.5

# Build pollinator -> bin membership from P matrix
# P_combined: (4515, 3895) — pollinators × bins
P_combined = pd.read_csv(
    "/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence_gbif_combined.csv",
    index_col=0
)

# For each bin, which pollinators are present?
print("Building bin -> pollinator mapping...")
bin_to_pollinators = {}
for bin_col in P_combined.columns:
    pols_in_bin = P_combined.index[P_combined[bin_col] == 1].tolist()
    # Only keep pollinators that are in our Vp
    pols_in_bin = [p for p in pols_in_bin if p in vp_idx and p in activity_curves.index]
    if len(pols_in_bin) > 0:
        bin_to_pollinators[bin_col] = pols_in_bin

print(f"Bins with at least one pollinator: {len(bin_to_pollinators):,}")

# Precompute Vf for our plant
vf_plant = Vf_arr[vf_idx[PLANT]]

# For each bin, compute A2 and per-week A3 predictions
print("Computing per-bin predictions...")

# Get all bins from achillea PPE data
achillea_week0 = achillea[achillea['week'] == 0][['cell_idx', 'centroid_lat', 'centroid_lon']].copy()
achillea_week0['bin'] = (
    ((achillea_week0['centroid_lat'] // BIN_SIZE) * BIN_SIZE).astype(str) + "_" +
    ((achillea_week0['centroid_lon'] // BIN_SIZE) * BIN_SIZE).astype(str)
)

results = []
for _, cell_row in achillea_week0.iterrows():
    bin_key = cell_row['bin']
    lat = cell_row['centroid_lat']
    lon = cell_row['centroid_lon']
    
    pols = bin_to_pollinators.get(bin_key, [])
    if len(pols) == 0:
        results.append({'lat': lat, 'lon': lon, 'bin': bin_key,
                       'pred_a2': np.nan,
                       **{f'pred_a3_w{w}': np.nan for w in [4,9,13,18]}})
        continue
    
    # A2: no temporal dimension
    a2_preds = []
    for pol in pols:
        # N = shared bins between plant and pollinator
        if PLANT in f_idx and pol in p_idx:
            N_val = int(F_arr[f_idx[PLANT]] @ P_arr[p_idx[pol]])
        else:
            N_val = 0
        feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [N_val]])
        a2_preds.append(clf_a2_subset.predict_proba(feat.reshape(1,-1))[0,1])
    
    # A3: week-specific delta
    a3_week_preds = {}
    for week in [4, 9, 13, 18]:
        week_preds = []
        for pol in pols:
            if PLANT in f_idx and pol in p_idx:
                N_val = int(F_arr[f_idx[PLANT]] @ P_arr[p_idx[pol]])
            else:
                N_val = 0
            f_w = flowering_curves.loc[PLANT, week] if week in flowering_curves.columns else 0
            a_w = activity_curves.loc[pol, week] if week in activity_curves.columns else 0
            delta = min(f_w, a_w)
            feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [N_val, delta]])
            week_preds.append(clf_a3.predict_proba(feat.reshape(1,-1))[0,1])
        a3_week_preds[f'pred_a3_w{week}'] = np.mean(week_preds)
    
    results.append({
        'lat': lat, 'lon': lon, 'bin': bin_key,
        'pred_a2': np.mean(a2_preds),
        **a3_week_preds
    })

pred_df = pd.DataFrame(results)
print(f"Done! Bins with predictions: {pred_df['pred_a2'].notna().sum():,}")
print(pred_df.describe())

In [ ]:
import pandas as pd

GLOBI_PATH = "/scratch/ariana.l/CfE2026CVforEcology/rawpollinatordata/interactions.csv.gz"
PLANT = 'Achillea millefolium'

KEEP_COLS = [
    'sourceTaxonName', 'targetTaxonName', 'interactionTypeName',
    'decimalLatitude', 'decimalLongitude', 'eventDate'
]

source_is_pollinator = {'visitsFlowersOf', 'pollinates', 'visits', 'hasFlowerVisitor'}
target_is_pollinator = {'flowersVisitedBy', 'pollinatedBy', 'visitedBy'}

BROAD_INTERACTION_TYPES = source_is_pollinator | target_is_pollinator
CONUS_LON = (-125, -66)
CONUS_LAT = (24, 50)

chunks = []
for chunk in pd.read_csv(GLOBI_PATH, usecols=KEEP_COLS, chunksize=100_000, low_memory=False):
    chunk = chunk[chunk['interactionTypeName'].isin(BROAD_INTERACTION_TYPES)]
    chunk = chunk.dropna(subset=['decimalLatitude', 'decimalLongitude'])
    chunk = chunk[
        chunk['decimalLongitude'].between(*CONUS_LON) &
        chunk['decimalLatitude'].between(*CONUS_LAT)
    ]
    # Filter to Achillea millefolium on either side
    chunk = chunk[
        (chunk['sourceTaxonName'] == PLANT) |
        (chunk['targetTaxonName'] == PLANT)
    ]
    if len(chunk) > 0:
        chunks.append(chunk)

achillea_globi = pd.concat(chunks, ignore_index=True)
achillea_globi['eventDate'] = pd.to_datetime(achillea_globi['eventDate'], errors='coerce')
achillea_globi = achillea_globi.dropna(subset=['eventDate'])
achillea_globi['month'] = achillea_globi['eventDate'].dt.month

print(f"Total Achillea records with dates: {len(achillea_globi):,}")
print(f"Month distribution:")
print(achillea_globi['month'].value_counts().sort_index())

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression

# --- Matrices ---
F = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_F_existence_phenofield.csv", index_col=0)
P_combined = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_P_existence_gbif_combined.csv", index_col=0)

F_arr = F.values
P_arr = P_combined.values
f_idx = {s: i for i, s in enumerate(F.index)}
p_idx = {s: i for i, s in enumerate(P_combined.index)}

# --- PCA embeddings ---
Vf_df = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vf_phenofield.csv", index_col=0)
Vp_df = pd.read_csv("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_Vp_gbif.csv", index_col=0)

Vf_arr = Vf_df.values
Vp_arr = Vp_df.values
vf_idx = {s: i for i, s in enumerate(Vf_df.index)}
vp_idx = {s: i for i, s in enumerate(Vp_df.index)}

# --- Models ---
with open("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_A3_logistic.pkl", "rb") as f:
    clf_a3 = pickle.load(f)

# A2 model — retrain on the fly from A3 training data (same split)
# If you saved clf_a2_subset separately, load it instead
# with open("/scratch/ariana.l/Stage 4 Link Prediction Model/stage4_A2_logistic.pkl", "rb") as f:
#     clf_a2_subset = pickle.load(f)

# --- PPE curves ---
flowering_curves = pd.read_parquet("/scratch/ariana.l/ppe-outputs/data/flowering_curves_used.parquet")
activity_curves = pd.read_parquet("/scratch/ariana.l/ppe-outputs/data/activity_curves_used.parquet")

# --- Achillea PPE opportunity surface ---
import glob
files = sorted(glob.glob("/scratch/ariana.l/ppe-outputs/opportunity_surface/part_*.parquet"))
achillea_data = []
for f in files:
    df = pd.read_parquet(f)
    subset = df[df['species'] == 'Achillea millefolium']
    if len(subset) > 0:
        achillea_data.append(subset)
achillea = pd.concat(achillea_data)

print("All variables loaded.")
print(f"Vf: {Vf_arr.shape}, Vp: {Vp_arr.shape}")
print(f"F: {F_arr.shape}, P: {P_arr.shape}")
print(f"flowering_curves: {flowering_curves.shape}, activity_curves: {activity_curves.shape}")
print(f"achillea: {achillea.shape}")

In [ ]:
import numpy as np
import pandas as pd

PLANT = 'Achillea millefolium'
BIN_SIZE = 0.5
WEEKS = list(range(1, 23))  # weeks 1–22 (Feb–May)

# Precompute vf for plant
vf_plant = Vf_arr[vf_idx[PLANT]]

print("Building bin -> pollinator mapping...")
bin_to_pollinators = {}
for bin_col in P_combined.columns:
    pols_in_bin = P_combined.index[P_combined[bin_col] == 1].tolist()
    pols_in_bin = [p for p in pols_in_bin if p in vp_idx and p in activity_curves.index]
    if pols_in_bin:
        bin_to_pollinators[bin_col] = pols_in_bin

print(f"Bins with at least one pollinator: {len(bin_to_pollinators):,}")

# Base grid from achillea PPE data (week-independent geometry)
achillea_week0 = achillea[achillea['week'] == 0][['cell_idx', 'centroid_lat', 'centroid_lon']].copy()
achillea_week0['bin'] = (
    ((achillea_week0['centroid_lat'] // BIN_SIZE) * BIN_SIZE).astype(str) + "_" +
    ((achillea_week0['centroid_lon'] // BIN_SIZE) * BIN_SIZE).astype(str)
)

# Precompute N and A2 pred per pollinator per bin (week-independent)
print("Precomputing A2 (week-independent)...")
bin_a2_cache = {}  # bin -> mean A2 pred
bin_pol_N_cache = {}  # bin -> list of (pol, N) tuples

for _, cell_row in achillea_week0.iterrows():
    bin_key = cell_row['bin']
    if bin_key in bin_a2_cache:
        continue
    pols = bin_to_pollinators.get(bin_key, [])
    if not pols:
        bin_a2_cache[bin_key] = np.nan
        bin_pol_N_cache[bin_key] = []
        continue

    pol_N_list = []
    a2_preds = []
    for pol in pols:
        N_val = int(F_arr[f_idx[PLANT]] @ P_arr[p_idx[pol]]) if PLANT in f_idx and pol in p_idx else 0
        feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [N_val]])
        a2_preds.append(clf_a2_subset.predict_proba(feat.reshape(1, -1))[0, 1])
        pol_N_list.append((pol, N_val))

    bin_a2_cache[bin_key] = np.mean(a2_preds)
    bin_pol_N_cache[bin_key] = pol_N_list

print("Computing A3 per week...")
results = []
for week in WEEKS:
    if week % 5 == 0:
        print(f"  week {week}/22...")
    for _, cell_row in achillea_week0.iterrows():
        bin_key = cell_row['bin']
        pol_N_list = bin_pol_N_cache.get(bin_key, [])
        pred_a2 = bin_a2_cache.get(bin_key, np.nan)

        if not pol_N_list:
            results.append({
                'lat': cell_row['centroid_lat'],
                'lon': cell_row['centroid_lon'],
                'bin': bin_key,
                'week': week,
                'pred_a2': np.nan,
                'pred_a3': np.nan
            })
            continue

        f_w = flowering_curves.loc[PLANT, week] if week in flowering_curves.columns else 0
        week_preds = []
        for pol, N_val in pol_N_list:
            a_w = activity_curves.loc[pol, week] if week in activity_curves.columns else 0
            delta = min(f_w, a_w)
            feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [N_val, delta]])
            week_preds.append(clf_a3.predict_proba(feat.reshape(1, -1))[0, 1])

        results.append({
            'lat': cell_row['centroid_lat'],
            'lon': cell_row['centroid_lon'],
            'bin': bin_key,
            'week': week,
            'pred_a2': pred_a2,
            'pred_a3': np.mean(week_preds)
        })

pred_df = pd.DataFrame(results)
pred_df.to_parquet(
    "/scratch/ariana.l/Stage 4 Link Prediction Model/pred_df_weeks1_22.parquet",
    index=False
)
print(f"Done! Shape: {pred_df.shape}")
print(pred_df.describe())